**General Description**

The following notebook contains the code to create, train, validate, and test a rainfall-runoff model using an LSTM network architecture. The notebook support running experiments in different large-sample hydrology datasets including: CAMELS-GB, CAMELS-US, CAMELS-DE. The details for each dataset can be read from a .yml file.

***Authors:***
- Eduardo Acuña Espinoza (eduardo.espinoza@kit.edu)
- Manuel Alvarez Chaves (manuel.alvarez-chaves@simtech.uni-stuttgart.de)

# Config initialization

In [1]:
# Import necessary packages
import datetime
import pickle
import random
import sys
import time
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from pathlib import Path

sys.path.append("..")
# Import classes and functions from other files
from hy2dl.datasetzoo import get_dataset
from hy2dl.evaluation.metrics import nse
from hy2dl.modelzoo import get_model
from hy2dl.training.loss import nse_basin_averaged
from hy2dl.utils.config import Config
from hy2dl.utils.optimizer import Optimizer
from hy2dl.utils.utils import set_random_seed, upload_to_device

# colorblind friendly palette
color_palette = {"observed": "#377eb8", "simulated": "#4daf4a"}

Part 1. Initialize information

In [2]:
# Create a dictionary where all the information will be stored
experiment_settings = {}

# Experiment name
# experiment_settings["experiment_name"] = "bs_256_uniqueBlocksTrue_random_0.8"
experiment_settings["experiment_name"] = "CPC_Generate_memmap_overlap_11.5m"

# paths to access the information
experiment_settings["path_data"] = "../data/CAMELS_DE"
experiment_settings["path_entities"] = "../data/basin_id/basins_camels_de_1582.txt"
# experiment_settings["path_entities"] = "../data/basin_id/basins_camels_de_hourly_3.txt"

experiment_settings["dynamic_input"] = [
    "precipitation_mean",
    "precipitation_stdev",
    "radiation_global_mean",
    "temperature_min",
    "temperature_max",
]

experiment_settings["target"] = ["discharge_spec_obs"]

# static attributes that will be used. If one is not using static_inputs, initialize the variable as an empty list.
experiment_settings["static_input"] = [
    "area",
    "elev_mean",
    "clay_0_30cm_mean",
    "sand_0_30cm_mean",
    "silt_0_30cm_mean",
    "artificial_surfaces_perc",
    "agricultural_areas_perc",
    "forests_and_seminatural_areas_perc",
    "wetlands_perc",
    "water_bodies_perc",
    "p_mean",
    "p_seasonality",
    "frac_snow",
    "high_prec_freq",
    "low_prec_freq",
    "high_prec_dur",
    "low_prec_dur",
]

# # # time periods 
experiment_settings["training_period"] = ["1970-10-01", "1998-12-31"] # 28 years for pre-training

# # # time periods (for short test)
# experiment_settings["training_period"] = ["2000-01-01", "2000-01-05"]
# experiment_settings["validation_period"] = ["2001-01-01", "2001-01-05"]
# experiment_settings["testing_period"] = ["2002-01-01", "2002-01-05"]

# model configuration
experiment_settings["hidden_size"] = 128
experiment_settings["batch_size_training"] = 2    # original: 256
experiment_settings["batch_size_evaluation"] = 1024 # original: 1024
experiment_settings["epochs"] = 3  # 12
experiment_settings["dropout_rate"] = 0.4
experiment_settings["learning_rate"] = 0.001
experiment_settings["steplr_step_size"] = 10
experiment_settings["steplr_gamma"] = 0.5
experiment_settings["validate_every"] = 1
experiment_settings["validate_n_random_basins"] = -1

experiment_settings["seq_length_hindcast"] = 365

# experiment_settings["dynamic_embedding"] = {"hiddens": [10, 10, 10]}

# This is set to sample training sample with a stride, to prevent samples overlaping
# 365 means two neighbor sample have a stride of 365 in timestep
experiment_settings["sample_stride"] = 15  # To do
experiment_settings["CPC_embedding"] = {"hiddens": [512, 512, 512, 512]}

# device to train the model
experiment_settings["device"] = "cuda:0"
# experiment_settings["device"] = "cpu"
experiment_settings["num_workers"] = 4  # ori: 4

# define random seed
experiment_settings["random_seed"] = 110

# dataset
experiment_settings["dataset"] = "camels_de"
# model
experiment_settings["model"] = "cpc_lstm" 
# experiment_settings["model"] = "CudaLSTM" 
experiment_settings["initial_forget_bias"] = 3.0

In [3]:
# Read experiment settings
config = Config(experiment_settings)
config.init_experiment()
config.dump()

Folder '../results/CPC_Generate_memmap_overlap_11.5m_seed_110' was created to store the results.


# Create pre-training datasets and dataloaders

In [4]:
# Get dataset class
Dataset = get_dataset(config)

# Dataset training
config.logger.info(f"Loading training data from {config.dataset} dataset")
total_time = time.time()

training_dataset = Dataset(cfg=config, time_period="training")

training_dataset.calculate_basin_std()
scaler = training_dataset.calculate_global_statistics(save_scaler=True)
training_dataset.standardize_data()

config.logger.info(f"Number of entities with valid samples: {len(training_dataset.df_ts)}")
config.logger.info(
    f"Time required to process {len(training_dataset.df_ts)} entities: "
    f"{datetime.timedelta(seconds=int(time.time() - total_time))}"
)
config.logger.info(f"Number of valid training samples: {len(training_dataset)}\n")

# Dataloader training
train_loader = DataLoader(
    dataset=training_dataset,
    batch_size=config.batch_size_training,
    shuffle=True,
    drop_last=True,
    collate_fn=training_dataset.collate_fn,
    num_workers=config.num_workers,
)

# Print details of a loader´s sample to check that the format is correct
config.logger.info("Details training dataloader".center(60, "-"))
config.logger.info(f"Batch structure (number of batches: {len(train_loader)})")
config.logger.info(f"{'Key':^30}|{'Shape':^30}")
# config.logger.info("-" * 60)
# Loop through the sample dictionary and print the shape of each element
for key, value in next(iter(train_loader)).items():
    if key.startswith(("x_d", "x_conceptual")):
        config.logger.info(f"{key}")
        for i, v in value.items():
            config.logger.info(f"{i:^30}|{str(v.shape):^30}")
    else:
        config.logger.info(f"{key:<30}|{str(value.shape):^30}")

config.logger.info("")  # prints a blank line

2025-12-08 17:25:11 - Loading training data from camels_de dataset


Processing entities: 100%|##########| 1582/1582 [01:54<00:00, 13.84entity/s]


2025-12-08 17:27:06 - Basins without valid samples in period of interest: ['DE112030', 'DE112130', 'DE112140', 'DE112160', 'DE112180', 'DE112200', 'DE112220', 'DE112260', 'DE112280', 'DE112310', 'DE112320', 'DE112330', 'DE112350', 'DE112360', 'DE112370', 'DE112380', 'DE112390', 'DE112400', 'DE112410', 'DE112420', 'DE112450', 'DE112480', 'DE112490', 'DE112500', 'DE211250', 'DE211270', 'DE211280', 'DE212450', 'DE212620', 'DE213630', 'DE214150', 'DE214260', 'DE214410', 'DE214430', 'DE214600', 'DE214700', 'DE214940', 'DE411270', 'DE411720', 'DE411790', 'DE411850', 'DE411920', 'DE412020', 'DE412060', 'DE412080', 'DE412230', 'DE810210', 'DE810220', 'DE810230', 'DE810280', 'DE810420', 'DE810440', 'DE810790', 'DE810800', 'DE810830', 'DE810840', 'DE810870', 'DE810950', 'DE811100', 'DE811110', 'DE811390', 'DE811420', 'DE812060', 'DE812170', 'DE910020', 'DE910500', 'DE911190', 'DEA10560', 'DEA10660', 'DEA10930', 'DEA11060', 'DEA11470', 'DEA11860', 'DEA11920', 'DEA11930', 'DEA12040', 'DEA12080', '

/software/all/jupyter/ai/2025-05-23/lib/python3.11/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


2025-12-08 17:27:08 - x_d
2025-12-08 17:27:08 -       precipitation_mean      |     torch.Size([2, 365])     
2025-12-08 17:27:08 -      precipitation_stdev      |     torch.Size([2, 365])     
2025-12-08 17:27:08 -     radiation_global_mean     |     torch.Size([2, 365])     
2025-12-08 17:27:08 -        temperature_min        |     torch.Size([2, 365])     
2025-12-08 17:27:08 -        temperature_max        |     torch.Size([2, 365])     
2025-12-08 17:27:08 - x_s                           |     torch.Size([2, 17])      
2025-12-08 17:27:08 - y_obs                         |    torch.Size([2, 1, 1])     
2025-12-08 17:27:08 - std_basin                     |    torch.Size([2, 1, 1])     
2025-12-08 17:27:08 - basin                         |             (2,)             
2025-12-08 17:27:08 - date                          |            (2, 1)            
2025-12-08 17:27:08 - 


# Show mean and std values in pre-training set

In [5]:
# Check the mean and std of all variables
with open(config.path_save_folder / "scaler.pickle", "rb") as f:
    scaler = pickle.load(f)
config.logger.info("The mean and std of all variables: %s", scaler)

2025-12-08 17:27:08 - The mean and std of all variables: {'x_d_mean': {'precipitation_mean': tensor(2.3211), 'precipitation_stdev': tensor(0.5260), 'radiation_global_mean': tensor(114.3739), 'temperature_min': tensor(4.2744), 'temperature_max': tensor(12.2574)}, 'x_d_std': {'precipitation_mean': tensor(4.7400), 'precipitation_stdev': tensor(1.0392), 'radiation_global_mean': tensor(85.1406), 'temperature_min': tensor(6.5823), 'temperature_max': tensor(8.4937)}, 'y_mean': tensor([0.9695]), 'y_std': tensor([1.5089]), 'x_s_mean': tensor([3.4417e+02, 3.2379e+02, 2.0969e+01, 3.6242e+01, 3.7619e+01, 6.7843e+00,
        5.6205e+01, 3.6387e+01, 1.6437e-01, 4.5867e-01, 2.3247e+00, 1.0354e-01,
        6.2630e-02, 1.6854e+01, 2.2697e+02, 1.1757e+00, 3.9619e+00]), 'x_s_std': tensor([8.1100e+02, 2.3799e+02, 7.4339e+00, 1.8901e+01, 1.1752e+01, 6.3241e+00,
        2.1817e+01, 2.2179e+01, 6.9588e-01, 1.7311e+00, 6.2471e-01, 1.4096e-01,
        3.2857e-02, 1.2861e+00, 1.6181e+01, 6.6101e-03, 6.5531e-02]

In [ ]:
# # Check the one train sample
# dataset_sample = training_dataset[3]
# print(f"One sample in training dataset look like: {dataset_sample}")

# Generate memmap.npy

In [6]:
# Choose mode: "pre_train" or "fine-tune"
mode = "pre_train"  # will generate data without labels
# mode = "fine_tune"  # will generate data with labels

# For my pc
# memmap_path = "./memmap.npy"
# meta_path = "./memmap_meta.npz"

# For haicore cluster
folder_path = Path("/hkfs/home/haicore/iwu/qa8171/Project/CPC2HY/CAMELS_DE/overlap_11.5m")
folder_path.mkdir(parents=True, exist_ok=True)

memmap_path = folder_path / "memmap.npy"
meta_path = folder_path / "memmap_meta.npz"

# generate memmap =====
total_time = time.time()
data_mm = np.memmap(
    memmap_path,
    dtype="float32",
    mode="w+",
    shape=((len(training_dataset) * config.seq_length_hindcast),
           (len(config.dynamic_input) + len(config.static_input)))
)
config.logger.info(
    "Time required to generate memmap.npy file: {}".format(
        datetime.timedelta(seconds=int(time.time() - total_time))))

start_list = []
length_list = []
file_idx_list = []
label_list = []

row_index = 0

meta_time = time.time()
for i in tqdm(range(len(training_dataset)), desc="Processing training samples", ncols=80):
    sample = training_dataset[i]

    # ===== x_d 动态变量 =====
    x_d_dict = sample["x_d"]
    dyn_keys = sorted(x_d_dict.keys())
    dyn_vars = [x_d_dict[k].numpy() for k in dyn_keys]
    x_d = np.stack(dyn_vars, axis=-1)  

    # ===== x_s 静态变量 =====
    x_s = sample["x_s"].numpy()
    x_s = np.tile(x_s, (config.seq_length_hindcast, 1))       

    # ===== 拼接 =====
    merged = np.concatenate([x_d, x_s], axis=-1) 

    # ===== 写入 memmap =====
    data_mm[row_index : row_index + config.seq_length_hindcast] = merged

    # ===== meta =====
    start_list.append(row_index)
    length_list.append(config.seq_length_hindcast)
    file_idx_list.append(0)

    # ===== 记录 label（y_obs） =====
    y_obs = sample["y_obs"].numpy().astype(np.float32)  # (N,)
    label_list.append(y_obs)

    row_index += config.seq_length_hindcast

data_mm.flush()

label_array = np.stack(label_list).astype(np.float32).squeeze()

# ===== Save meta =====
if mode=="pre_train":
    np.savez_compressed(
        meta_path,
        start=np.array(start_list),
        length=np.array(length_list),
        shape=np.array([[(len(training_dataset) * config.seq_length_hindcast), (len(config.dynamic_input) + len(config.static_input))]]),
        file_idx=np.array(file_idx_list),
        dtype=np.array("float32"),
        filenames=np.array([memmap_path])
    )
elif mode=="fine_tune":
    np.savez_compressed(
        meta_path,
        start=np.array(start_list),
        length=np.array(length_list),
        shape=np.array([[(len(training_dataset) * config.seq_length_hindcast), (len(config.dynamic_input) + len(config.static_input))]]),
        file_idx=np.array(file_idx_list),
        dtype=np.array("float32"),
        filenames=np.array([memmap_path]),
        label=label_array
        )
config.logger.info(
    "Time required to generate meta file: {}".format(
        datetime.timedelta(seconds=int(time.time() - meta_time))))

2025-12-08 17:27:08 - Time required to generate memmap.npy file: 0:00:00


Processing training samples: 100%|████| 761957/761957 [02:33<00:00, 4953.78it/s]


2025-12-08 17:29:44 - Time required to generate meta file: 0:02:35


In [7]:
# 1. 加载元数据
meta = np.load(meta_path, allow_pickle=True)
start = meta["start"]
length = meta["length"]
shape = meta["shape"]
file_idx = meta["file_idx"]
dtype = np.dtype(str(meta["dtype"]))
config.logger.info("dataset shape: %s", shape)  # the first element is total length (sequence lenghth * num_samples), the second is feature dimension (12 for ECG)

# 2. 打开 memmap 文件
memmap_data = np.memmap(memmap_path, dtype=dtype, mode='r', shape=tuple(shape[0]))

# # 3. 访问第一个样本（示例）
idx = 0
sample_start = start[idx]
sample_length = length[idx]
config.logger.info("No. samples: %s", len(start))

sample = memmap_data[sample_start : sample_start + sample_length]
config.logger.info("One sample shape: %s", sample.shape)
config.logger.info("The first 3 timesteps of this sample: %s", sample[:3, :])  # 显示前5个时间步的12维数据

2025-12-08 17:29:44 - dataset shape: [[278114305        22]]
2025-12-08 17:29:44 - No. samples: 761957
2025-12-08 17:29:44 - One sample shape: (365, 22)
2025-12-08 17:29:44 - The first 3 timesteps of this sample: [[-0.43270838 -0.29448983 -0.3647363   0.1204008   0.03426883  0.5165246
   2.0944397   1.0655102  -0.60591906  0.30301216  0.16693892 -0.5479837
   0.50151074  0.10867604 -0.16675273  1.0330398  -0.6635961   1.7460427
  -1.2624671  -1.5018715   2.1707652  -3.8442175 ]
 [ 1.0736083   3.9201612  -1.0423214  -0.01381681  0.27278566  0.5165246
   2.0944397   1.0655102  -0.60591906  0.30301216  0.16693892 -0.5479837
   0.50151074  0.10867604 -0.16675273  1.0330398  -0.6635961   1.7460427
  -1.2624671  -1.5018715   2.1707652  -3.8442175 ]
 [ 0.4596893   1.2547313  -0.6930168  -0.22338486 -0.14347938  0.5165246
   2.0944397   1.0655102  -0.60591906  0.30301216  0.16693892 -0.5479837
   0.50151074  0.10867604 -0.16675273  1.0330398  -0.6635961   1.7460427
  -1.2624671  -1.5018715   2

In [8]:
config.logger.info("Keys in meta file: %s", meta.files)
config.logger.info("Detailed contents:")
for key in meta.files:
    config.logger.info(f"{key}: %s", meta[key])

2025-12-08 17:29:44 - Keys in meta file: ['start', 'length', 'shape', 'file_idx', 'dtype', 'filenames']
2025-12-08 17:29:44 - Detailed contents:
2025-12-08 17:29:44 - start: [        0       365       730 ... 278113210 278113575 278113940]
2025-12-08 17:29:44 - length: [365 365 365 ... 365 365 365]
2025-12-08 17:29:44 - shape: [[278114305        22]]
2025-12-08 17:29:44 - file_idx: [0 0 0 ... 0 0 0]
2025-12-08 17:29:44 - dtype: float32
2025-12-08 17:29:44 - filenames: [PosixPath('/hkfs/home/haicore/iwu/qa8171/Project/CPC2HY/CAMELS_DE/overlap_11.5m/memmap.npy')]
